In [ ]:
from huggingface_hub import InferenceClient
import os

In [ ]:
# Import de la clef d'API (via variable d'environnement) et du modèle
api_key = os.getenv("HF_API_KEY")
modelID="meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
#Extraction des différents prompts depuis le fichier texte
filename = "prompt_syst.txt"
prompts = {}
current_key = None
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line.startswith('[') and line.endswith(']'):
            current_key = line[1:-1]
            prompts[current_key] = ""
        elif current_key and line:
            prompts[current_key] += line + " "

Tu es l'assistant virtuel d'information médicale de Tessan. PÉRIMÈTRE : Fournis uniquement des informations médicales générales et fiables[cite: 14]. INTERDITS : - Ne pose JAMAIS de diagnostic ("Vous avez une grippe")[cite: 15]. - Ne prescris JAMAIS de médicament[cite: 15]. - N'interprète JAMAIS d'examens médicaux[cite: 15]. STYLE : Sois clair, professionnel, rassurant et non alarmiste[cite: 15]. URGENCES : Si l'utilisateur évoque des Red Flags (douleur poitrine, étouffement, paralysie, perte de connaissance), ordonne-lui immédiatement d'appeler le 15 ou de se rendre aux urgences[cite: 16]. 


In [ ]:
class Discussion ():
    def __init__(self, modelID, api_key, prompt_sys, window=5, max_token=100, temp=0.2):
        self.client = InferenceClient(api_key=api_key)
        self.modelID = modelID
        self.window = window #longueur de la fenêtre glissante de mémoire
        self.max_token = max_token #longueur de la réponse de l'IA
        self.temp = temp
        self.hist = []
        self.intent = None
        
        #Définition des prompts système en fonction du type de discussion
        self.prompt_sys = prompt_sys

    def getlog(self):
        return self.hist
    
    def callAI(self, context):
        answer_gen = self.client.chat.completions.create(
                model=self.modelID,
                messages=context,
                max_tokens=self.max_token,
                temperature=self.temp)
        return answer_gen.choices[0].message.content
    
    def process_input(self, user_input):
        self.hist.append({"role":"user","content":user_input}) #Ajout de l'entrée de l'utilisateur à l'historique
        
        #Test de l'intention si elle n'a pas encore été détectée
        if self.intent is None : 
            print("Détection de l'intention")
            context_detection = [{"role":"system","content":self.prompt_sys["INTENT"]}] + self.hist[-1:]
            intent = self.callAI(context_detection).upper()

            if "MEDICAL" in intent: self.intent = "MEDICAL"
            elif "ADMIN" in intent: self.intent = "ADMIN"
            else: self.intent = "AUTRE"
            print(self.intent)
        
        #Réponse à la requete en fonction de l'intention
        if self.intent == "AUTRE":
            answer_fin= self.prompt_sys["AUTRE"]
        else :
            #Créer la fenêtre de contexte :
            context = [{"role":"system","content":self.prompt_sys[self.intent]}] + self.hist[-self.window:]
            
            #Call l'IA pour répondre :
            answer_fin = self.callAI(context)
            if self.intent == "MEDICAL" :
                answer_fin += "\n\nSouhaitez-vous passer en téléconsultation maintenant ?"
            
        self.hist.append({"role":"assistant","content":answer_fin})

        return answer_fin


In [ ]:
chat = Discussion(modelID=modelID,api_key=api_key)

In [ ]:
user_input=input()

In [ ]:
print(chat.process_input(user_input=user_input))

In [ ]:
user_input = input()

In [ ]:
print(chat.process_input(user_input=user_input))